[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/12_montecarlo/notebooks/aplicaciones/05_caminata_aleatoria.ipynb)

# Aplicación 5: Caminata Aleatoria y Difusión

## La conexión histórica

En 1946, el problema que motivó a Stanislaw Ulam era precisamente este: **cómo se mueven los neutrones** a través de un material fisible.

Un neutrón entra al material, choca con un núcleo, cambia de dirección de forma aleatoria, choca de nuevo, se absorbe o escapa. La pregunta: ¿cuántos neutrones liberan más neutrones? ¿Qué fracción escapa? ¿Cuántos son absorbidos?

Resolver esto analíticamente requería integrar ecuaciones de difusión en geometrías complejas. Monte Carlo lo resuelve directamente: **simula el camino de cada neutrón**.

El modelo matemático subyacente es la **caminata aleatoria**: el proceso más simple que captura movimiento estocástico.

In [ ]:
# Descomenta si estás en Colab
# !pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

rng = np.random.default_rng(42)
print("Listo ✓")

---
## Parte 1: Caminata Aleatoria en 1D

### Definición

Una partícula empieza en $X_0 = 0$. En cada paso:
- Con probabilidad $p$: da un paso hacia la derecha ($+1$)
- Con probabilidad $1-p$: da un paso hacia la izquierda ($-1$)

Después de $n$ pasos: $X_n = \sum_{i=1}^n \xi_i$ donde $\xi_i \in \{-1, +1\}$.

Para $p = 1/2$ (caminata simétrica):
- $\mathbb{E}[X_n] = 0$ (sin deriva)
- $\text{Var}[X_n] = n$ (crece linealmente)
- Distancia típica: $|X_n| \approx \sqrt{n}$ — el famoso escalamiento raíz cuadrada

In [ ]:
def random_walk_1d(n_steps, n_walks, p=0.5, seed=None):
    """Simulate n_walks 1D random walks of n_steps each."""
    rng_local = np.random.default_rng(seed)
    steps = rng_local.choice([-1, 1], size=(n_walks, n_steps),
                              p=[1-p, p])
    positions = np.hstack([np.zeros((n_walks, 1)), np.cumsum(steps, axis=1)])
    return positions

n_steps = 1000
n_walks = 20
positions = random_walk_1d(n_steps, n_walks, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Individual walks
for i in range(n_walks):
    ax1.plot(positions[i], lw=0.7, alpha=0.6)
ax1.axhline(0, color="black", lw=1.5, ls="--")
ax1.set_xlabel("Paso")
ax1.set_ylabel("Posición $X_n$")
ax1.set_title(f"{n_walks} caminatas aleatorias en 1D")

# Verify E[X_n] = 0 and Var[X_n] = n
n_large = 10_000
pos_large = random_walk_1d(n_steps, n_large, seed=42)
ns = np.arange(n_steps + 1)

ax2.plot(ns, pos_large.mean(axis=0), color="#E94F37", lw=2,
         label="$\\mathbb{E}[X_n]$ empírico")
ax2.plot(ns, np.sqrt(pos_large.var(axis=0)), color="#2E86AB", lw=2,
         label="$\\sqrt{\\text{Var}[X_n]}$ empírico")
ax2.plot(ns, np.zeros_like(ns), "r--", lw=1, label="$\\mathbb{E}[X_n] = 0$ (teórico)")
ax2.plot(ns, np.sqrt(ns), "b--", lw=1, label="$\\sqrt{n}$ (teórico)")
ax2.set_xlabel("Paso $n$")
ax2.set_title("Verificación: $E[X_n]=0$, $\\text{Std}[X_n]=\\sqrt{n}$")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Quantitative check
print(f"n={n_steps}: E[X_n] = {pos_large[:, -1].mean():.3f} (esperado: 0)")
print(f"n={n_steps}: Std[X_n] = {pos_large[:, -1].std():.2f} (esperado: {np.sqrt(n_steps):.2f})")

---
## Parte 2: Conexión con la Ecuación de Calor

### La distribución de la posición converge a una Gaussiana

Por el **Teorema Central del Límite**, la posición $X_n$ después de $n$ pasos tiene distribución:

$$X_n \xrightarrow{d} \mathcal{N}(0, n)$$

Esto no es solo un resultado estadístico — es la conexión profunda con la **ecuación de calor**:

$$\frac{\partial u}{\partial t} = \frac{1}{2}\frac{\partial^2 u}{\partial x^2}$$

Si una "nube de partículas" empieza concentrada en $x=0$, su distribución en tiempo $t$ es exactamente $\mathcal{N}(0, t)$. La caminata aleatoria **ES** difusión.

Este fue el insight central para Los Álamos: simular partículas individuales con pasos aleatorios equivale a resolver la ecuación de difusión.

In [ ]:
# Show that the distribution of X_n is Gaussian
n_large = 50_000
ns_show = [10, 50, 200, 1000]  # <-- CHANGE THIS

fig, axes = plt.subplots(1, len(ns_show), figsize=(4*len(ns_show), 4), sharey=False)

for ax, n in zip(axes, ns_show):
    pos = random_walk_1d(n, n_large, seed=42)[:, -1]

    # Histogram
    ax.hist(pos, bins=50, density=True, color="#2E86AB", alpha=0.7, edgecolor="white")

    # Theoretical Gaussian
    x_range = np.linspace(pos.min(), pos.max(), 300)
    ax.plot(x_range, norm.pdf(x_range, 0, np.sqrt(n)),
            "r-", lw=2, label=f"$\\mathcal{{N}}(0, {n})$")
    ax.set_title(f"$n = {n}$")
    ax.set_xlabel("$X_n$")
    ax.legend(fontsize=8)

fig.suptitle("Distribución de $X_n$: converge a $\\mathcal{N}(0, n)$ (ecuación de calor)",
             y=1.02)
plt.tight_layout()
plt.show()

---
## Parte 3: Caminata Aleatoria en 2D

En 2D, en cada paso la partícula se mueve en una de cuatro direcciones con igual probabilidad:
Norte, Sur, Este, Oeste.

La distancia al origen después de $n$ pasos sigue escalando como $\sqrt{n}$, pero ahora en el plano.

In [ ]:
def random_walk_2d(n_steps, n_walks, seed=None):
    """Simulate 2D random walks on the lattice (4 directions)."""
    rng_local = np.random.default_rng(seed)
    directions = np.array([[1,0], [-1,0], [0,1], [0,-1]])
    idx = rng_local.integers(0, 4, size=(n_walks, n_steps))
    steps = directions[idx]  # (n_walks, n_steps, 2)
    positions = np.concatenate(
        [np.zeros((n_walks, 1, 2)), np.cumsum(steps, axis=1)], axis=1
    )
    return positions

n_steps_2d = 2000
n_walks_2d = 10
walks_2d = random_walk_2d(n_steps_2d, n_walks_2d, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Show walks
colors_2d = plt.cm.tab10(np.linspace(0, 1, n_walks_2d))
for i in range(n_walks_2d):
    ax1.plot(walks_2d[i, :, 0], walks_2d[i, :, 1],
             lw=0.5, alpha=0.6, color=colors_2d[i])
    ax1.plot(*walks_2d[i, 0], "o", ms=5, color=colors_2d[i])
    ax1.plot(*walks_2d[i, -1], "s", ms=7, color=colors_2d[i])
ax1.plot(0, 0, "k*", ms=15, label="Origen")
ax1.set_aspect("equal")
ax1.set_title(f"Caminatas 2D ({n_steps_2d} pasos)\n● inicio · ■ final")
ax1.set_xlabel("$x$")
ax1.set_ylabel("$y$")
ax1.legend()

# Distance scaling
n_large_2d = 5_000
walks_large = random_walk_2d(n_steps_2d, n_large_2d, seed=42)
distances = np.sqrt((walks_large[:, :, 0]**2 + walks_large[:, :, 1]**2))
mean_dist = distances.mean(axis=0)
ns_2d = np.arange(n_steps_2d + 1)

ax2.plot(ns_2d, mean_dist, color="#2E86AB", lw=1.5, label="$\\mathbb{E}[|X_n|]$ empírico")
ax2.plot(ns_2d, np.sqrt(ns_2d * np.pi / 2), "--", color="#E94F37", lw=1.5,
         label="$\\sqrt{\\pi n/2}$ (teórico)")
ax2.set_xlabel("Paso $n$")
ax2.set_ylabel("Distancia media al origen")
ax2.set_title("Escalamiento $\\sqrt{n}$ en 2D")
ax2.legend()

plt.tight_layout()
plt.show()

---
## Parte 4: Tiempo de Absorción

Pregunta: **¿cuántos pasos tarda la partícula en llegar a la frontera?**

Modelo: la partícula empieza en $x=0$ y se absorbe cuando $|X_n| \geq L$ (toca la pared).

El **primer tiempo de llegada** $\tau_L = \min\{n : |X_n| \geq L\}$ tiene una distribución interesante.

Para la caminata en $\mathbb{Z}$: $\mathbb{E}[\tau_L] = L^2$ (el tiempo escala con $L^2$, consistente con $\sqrt{n}$ de distancia).

In [ ]:
def first_hitting_time(L, n_walks, max_steps=100_000, seed=None):
    """First time |X_n| >= L for 1D random walks."""
    rng_local = np.random.default_rng(seed)
    hitting_times = []

    for _ in range(n_walks):
        x = 0
        for t in range(1, max_steps + 1):
            x += rng_local.choice([-1, 1])
            if abs(x) >= L:
                hitting_times.append(t)
                break
        else:
            hitting_times.append(max_steps)  # didn't hit within max_steps

    return np.array(hitting_times)

# More efficient vectorized version
def first_hitting_time_vec(L, n_walks, seed=None):
    """Vectorized first hitting time for |X_n| >= L."""
    rng_local = np.random.default_rng(seed)
    max_steps = 10 * L**2
    steps = rng_local.choice([-1, 1], size=(n_walks, max_steps))
    positions = np.cumsum(steps, axis=1)
    # Find first time |X| >= L for each walk
    crossed = np.abs(positions) >= L
    hitting = np.argmax(crossed, axis=1) + 1  # +1 for 1-indexing
    # Handle walks that never crossed
    never = ~crossed.any(axis=1)
    hitting[never] = max_steps
    return hitting

Ls = [5, 10, 20]  # <-- CHANGE THIS
n_walks = 5_000

fig, axes = plt.subplots(1, len(Ls), figsize=(5*len(Ls), 4))
for ax, L in zip(axes, Ls):
    ht = first_hitting_time_vec(L, n_walks, seed=42)
    ax.hist(ht, bins=50, color="#2E86AB", alpha=0.7, edgecolor="white", density=True)
    ax.axvline(ht.mean(), color="#E94F37", ls="--", lw=2,
               label=f"Media = {ht.mean():.0f}")
    ax.axvline(L**2, color="black", ls=":", lw=1.5,
               label=f"$L^2 = {L**2}$ (teórico)")
    ax.set_title(f"$L = {L}$")
    ax.set_xlabel("Tiempo de absorción")
    ax.legend(fontsize=9)

fig.suptitle("Distribución del primer tiempo de llegada $\\tau_L$\n$\\mathbb{{E}}[\\tau_L] \\approx L^2$",
             y=1.03)
plt.tight_layout()
plt.show()

---
## Parte 5: El Puzzle de Pólya

### Una pregunta sorprendente

**¿Una caminata aleatoria regresa al origen con probabilidad 1?**

La respuesta depende de la dimensión, y el resultado es sorprendente:

| Dimensión | $P(\text{regreso})$ | Resultado |
|:---------:|:-------------------:|----------|
| $d = 1$ | $1$ | Recurrente — ¡siempre regresa! |
| $d = 2$ | $1$ | Recurrente — ¡siempre regresa! |
| $d = 3$ | $\approx 0.3405$ | **Transitoria** — puede irse para siempre |
| $d \geq 3$ | $< 1$ | Transitoria |

**Teorema de Pólya (1921)**: la caminata aleatoria en $\mathbb{Z}^d$ es recurrente si y solo si $d \leq 2$.

¿Por qué? En 1D y 2D, el espacio es "suficientemente pequeño" para que la partícula eventualmente revisite cualquier punto. En 3D, hay "demasiado espacio" y la partícula puede escapar.

In [ ]:
def return_probability(d, n_steps, n_walks, seed=None):
    """Estimate P(walk returns to origin in d dimensions within n_steps)."""
    rng_local = np.random.default_rng(seed)
    returned = 0

    for _ in range(n_walks):
        pos = np.zeros(d, dtype=int)
        returned_this = False
        for _ in range(n_steps):
            # Random step: choose dimension and direction
            dim = rng_local.integers(0, d)
            direction = rng_local.choice([-1, 1])
            pos[dim] += direction
            if np.all(pos == 0):
                returned_this = True
                break
        returned += returned_this

    return returned / n_walks

# Theoretical values
# d=1: 1.0
# d=2: 1.0
# d=3: 0.3405 (Watson 1939)

n_steps = 10_000  # <-- CHANGE THIS (more steps → better estimate for d=1,2)
n_walks = 500    # <-- CHANGE THIS (more walks → smaller std error)

print("Estimando probabilidad de regreso al origen...")
results_polya = {}
theoretical = {1: 1.0, 2: 1.0, 3: 0.3405}

for d in [1, 2, 3]:
    p_return = return_probability(d, n_steps, n_walks, seed=42)
    results_polya[d] = p_return
    theo = theoretical[d]
    print(f"  d={d}: P(retorno) ≈ {p_return:.4f}  (teórico: {theo})")

print(f"\n(Para d=1,2: con n_steps finito, obtenemos una cota inferior de 1.0)")
print(f"(Para d=3: deberías obtener ~0.34)")

In [ ]:
# Visualize convergence of return probability estimate as n_steps increases
steps_range = [100, 500, 1000, 2000, 5000, 10_000, 20_000]
n_walks_v = 300  # fewer walks for speed

probs_by_dim = {d: [] for d in [1, 2, 3]}
print("Calculando... (puede tomar un momento)")

for n_s in steps_range:
    for d in [1, 2, 3]:
        p = return_probability(d, n_s, n_walks_v, seed=d*1000+n_s)
        probs_by_dim[d].append(p)

fig, ax = plt.subplots(figsize=(10, 5))
colors_d = {1: "#2E86AB", 2: "#27AE60", 3: "#E94F37"}
for d in [1, 2, 3]:
    ax.semilogx(steps_range, probs_by_dim[d], "o-",
                color=colors_d[d], lw=2, ms=7, label=f"$d={d}$")
    ax.axhline(theoretical[d], color=colors_d[d], ls=":", lw=1, alpha=0.7)

ax.set_xlabel("Número máximo de pasos")
ax.set_ylabel("$P$(regreso al origen)")
ax.set_title("Teorema de Pólya: recurrencia de la caminata aleatoria por dimensión\n"
             "(líneas punteadas: valores teóricos)")
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.show()

print("\n¿Por qué cambia con la dimensión?")
print("En d=1 y d=2: el espacio es 'suficientemente pequeño', la caminata cubre todo.")
print("En d=3: hay demasiado espacio. La partícula puede escapar al infinito.")

---
## 🔧 Ejercicio 1: La Ruina del Jugador

Un jugador empieza con $k$ fichas. En cada ronda:
- Gana 1 ficha con probabilidad $p$
- Pierde 1 ficha con probabilidad $1-p$

El juego termina cuando llega a $N$ fichas (gana) o a $0$ fichas (se arruina).

**Resultado analítico** (para $p \neq 1/2$):

$$P(\text{arruinarse} \mid X_0 = k) = \frac{(q/p)^k - (q/p)^N}{1 - (q/p)^N}, \qquad q = 1-p$$

Para $p = 1/2$: $P(\text{arruinarse}) = 1 - k/N$.

Verifica esta fórmula con Monte Carlo y experimenta:
1. ¿Cuánto cambia la probabilidad de arruinarse si $p = 0.49$ vs $p = 0.51$?
2. ¿Es "justo" un casino donde el house tiene ventaja del 1% ($p_{\text{jugador}} = 0.49$)?

In [ ]:
def gamblers_ruin(k, N, p, n_games, seed=None):
    """Simulate Gambler's Ruin. Returns fraction of games where player goes broke."""
    rng_local = np.random.default_rng(seed)
    ruin_count = 0
    for _ in range(n_games):
        x = k
        while 0 < x < N:
            x += 1 if rng_local.random() < p else -1
        if x == 0:
            ruin_count += 1
    return ruin_count / n_games

def gamblers_ruin_exact(k, N, p):
    """Exact probability of ruin starting from k."""
    if abs(p - 0.5) < 1e-10:  # fair game
        return 1 - k / N
    q = 1 - p
    r = q / p
    return (r**k - r**N) / (1 - r**N)

k = 50   # starting capital  # <-- CHANGE THIS
N = 100  # target            # <-- CHANGE THIS
n_games = 2_000
ps = [0.45, 0.48, 0.49, 0.50, 0.51, 0.52, 0.55]  # <-- CHANGE THIS

print(f"Gambler's Ruin: k={k}, N={N}")
print(f"{'p':>6} | {'MC':>10} | {'Exacto':>10} | {'Diferencia':>12}")
print("-" * 45)
for p in ps:
    mc_p = gamblers_ruin(k, N, p, n_games, seed=int(p*100))
    exact_p = gamblers_ruin_exact(k, N, p)
    print(f"{p:>6.3f} | {mc_p:>10.4f} | {exact_p:>10.4f} | {abs(mc_p-exact_p):>12.4f}")

print(f"\nCon p=0.49 (casino 1% ventaja): P(arruina) = {gamblers_ruin_exact(k, N, 0.49):.4f}")
print(f"Con p=0.50 (juego justo):       P(arruina) = {gamblers_ruin_exact(k, N, 0.50):.4f}")

---
## 🔧 Ejercicio 2: Paso Asimétrico

Hasta ahora asumimos $p = 0.5$ (pasos simétricos). ¿Qué pasa con $p \neq 0.5$?

Para $p \neq 1/2$:
- $\mathbb{E}[X_n] = n(2p-1)$ — hay **deriva** (drift)
- $\text{Var}[X_n] = 4p(1-p)n$ — la varianza cambia

1. Simula caminatas con $p \in \{0.3, 0.5, 0.7\}$ y muestra las trayectorias
2. Verifica que la distancia esperada escala como $|2p-1| \cdot n$ para $p \neq 0.5$
3. **Conexión**: ¿cómo se conecta esto con la ecuación de advección-difusión $\partial_t u = -v \partial_x u + D \partial_{xx} u$?

In [ ]:
n_steps_asym = 500
n_walks_asym = 5_000
ps_asym = [0.3, 0.5, 0.7]  # <-- CHANGE THIS

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for p_val in ps_asym:
    pos = random_walk_1d(n_steps_asym, n_walks_asym, p=p_val, seed=int(p_val*100))
    mean_pos = pos.mean(axis=0)
    ns_asym = np.arange(n_steps_asym + 1)
    drift = (2*p_val - 1) * ns_asym

    axes[0].plot(ns_asym, mean_pos, lw=1.5, label=f"$p={p_val}$ (empírico)")
    axes[0].plot(ns_asym, drift, "--", lw=1, alpha=0.7)

axes[0].set_xlabel("Paso $n$")
axes[0].set_ylabel("$\\mathbb{E}[X_n]$")
axes[0].set_title("Media de la caminata\n(líneas punteadas: $n(2p-1)$ teórico)")
axes[0].legend(fontsize=9)

# Show final distribution for each p
for p_val, color in zip(ps_asym, ["#2E86AB", "#27AE60", "#E94F37"]):
    pos = random_walk_1d(n_steps_asym, n_walks_asym, p=p_val, seed=int(p_val*100))
    final_pos = pos[:, -1]
    mu_theory = n_steps_asym * (2*p_val - 1)
    sigma_theory = np.sqrt(4 * p_val * (1-p_val) * n_steps_asym)
    axes[1].hist(final_pos, bins=60, density=True, alpha=0.4, color=color,
                 label=f"$p={p_val}$")
    x_range = np.linspace(final_pos.min(), final_pos.max(), 200)
    axes[1].plot(x_range, norm.pdf(x_range, mu_theory, sigma_theory),
                 "-", color=color, lw=2)

axes[1].set_xlabel("$X_n$")
axes[1].set_ylabel("Densidad")
axes[1].set_title(f"Distribución de $X_{{n={n_steps_asym}}}$\n(sólidas: teórico $\\mathcal{{N}}(n(2p-1), 4p(1-p)n)$)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Reflexión: ¿Por qué importa esto?

La caminata aleatoria aparece en:

| Dominio | Proceso |
|---------|--------|
| **Física** | Difusión de partículas (el problema original de Los Álamos) |
| **Finanzas** | Movimiento Browniano Geométrico (modelo de precios) |
| **Biología** | Movimiento de bacterias (*E. coli* usa "run and tumble") |
| **Computer Science** | Random walks en grafos (PageRank, algoritmos de muestreo) |
| **Estadística** | MCMC: la cadena de Markov es una caminata aleatoria diseñada |

El teorema de Pólya no es solo una curiosidad matemática: nos dice que en dimensiones bajas ($d \leq 2$), cualquier proceso de difusión eventualmente "visita" todo el espacio. En dimensiones altas, puede escapar — y eso tiene implicaciones para el diseño de algoritmos de exploración.

La conexión más profunda: **MCMC es una caminata aleatoria diseñada para explorar distribuciones de probabilidad**. El diseño del paso (el kernel de transición) determina si la cadena mezcla rápida o lentamente, y si el análogo del teorema de Pólya aplica en el espacio de parámetros.